In [1]:
import sys
sys.path.append("../")

In [2]:
import json
import os
import pandas as pd

from src.database_utils.db_info import get_preprocessed_data
from src.database_utils.evaluate import get_snowflake_sql_result, compare_pandas_table

In [3]:
instane_id = "sf_bq012"

preprocessed_data = get_preprocessed_data(instane_id)
sql_query = preprocessed_data["query"]
database_id = preprocessed_data["db_id"]
is_save = True
save_dir = None
file_name = "result.csv"

In [4]:
status, df = get_snowflake_sql_result(sql_query, database_id, is_save=False, save_dir="./", file_name="result.csv")
df

,average_balance_trillion
0,26327002.23


In [5]:
compare_pandas_table(df, df)

condition_cols []
The status of our query is True.
The status of generated query is True.
The status of generated query without group by is True.
condition_cols []
The result of comparing our query and generated query is 0.
condition_cols []
The result of comparing our query and generated query without group by is 0.
condition_cols []
The result of comparing our query and generated query is 0.
condition_cols []
The result of comparing our query and generated query without group by is 0.
condition_cols []
condition_cols []
condition_cols []
The status of our query is True.
The status of generated query is True.
The status of generated query without group by is True.
condition_cols []
condition_cols []
condition_cols []
condition_cols []
condition_cols []
The status of our query is True.
The status of generated query is True.
The status of generated query without group by is True.
condition_cols []
condition_cols []
condition_cols []


1

In [7]:
dag_path = "/Users/stalaei/Desktop/Projects/Spider2/logs/decomposed_sqls/test/dag_sf_bq012.json"
dag = json.load(open(dag_path, 'r', encoding='utf-8'))
dag[0].keys()

dict_keys(['id', 'description', 'equivalent_natural_question', 'sql_query', 'dag_dependencies'])

In [8]:
from IPython.display import display

# Force output to display in notebook cell
from IPython.display import display, Markdown

for node in dag:
    # Use display() instead of print for notebook output
    display(Markdown(f"""
**Id:** {str(node.get('id', 'No ID found'))}

**Description:** {str(node.get('description', 'No description found'))}

**Results:**
"""))
    
    # Add error handling for missing sql_query
    if "sql_query" not in node:
        display(Markdown("Error: No SQL query found in node"))
        continue
        
    status, df = get_snowflake_sql_result(node["sql_query"], database_id, is_save=False)
    if status:
        display(df.head())
    else:
        display(Markdown(f"Error executing query: {df}"))
    display(Markdown("---"))


**Id:** A

**Description:** Filters the Ethereum TRACES table for successful transactions (status = 1) with to_address not null, excluding call types (delegatecall, callcode, staticcall). Each qualifying row represents Ether credited to the to_address.

**Results:**


,address,value
0,0xea674fdde714fd979de3edf0f56aa9716b898ec8,5000000000000000000.000000000
1,0xb2930b35844a230f00e51431acae96fe543a0347,5000000000000000000.000000000
2,0x2a65aca4d5fc5b5c859090a6c34d164135398226,3000000000000000000.000000000
3,0x52bc44d5378309ee2abf1539bf71de1b7d7be3b5,5000000000000000000.000000000
4,0xea674fdde714fd979de3edf0f56aa9716b898ec8,5000000000000000000.000000000


---


**Id:** B

**Description:** Similar filter on the TRACES table for successful transactions, but focusing on the from_address. Each qualifying row represents Ether debited from the from_address (i.e., how much Ether the sender transferred out).

**Results:**


,address,value
0,0x6ca859d4f9a632ddf99d405b826409fd8a61539d,0E-9
1,0xe27d5876d71c2a3dbaf9a8394c031869b3ae1dd7,0E-9
2,0x094e5dca672bdca248e1a52bd9c15d7549907962,0E-9
3,0x4c3e3ec831e6f48a4d51f43bc54df64c26b116d7,0E-9
4,0x99f2b16a93705e464cbbdad7e515e0ec97770dbb,0E-9


...

,address,balance
0,0x7727e5113d1d161373623e5f49fd568b4f543a9e,38117709844226507899655.000000000
1,0x209c4784ab1e8183cf58ca33cb740efbf3fc18ef,35474695192161340660664.000000000
2,0xcff5a79f5d2dfc8b10569c6bb105194cb87a10be,28747477551000000000000.000000000
3,0x22b84d5ffea8b801c0422afe752377a64aa738c2,27030229490926374307329.000000000
4,0x93025150b13eb744d51d927549a30630912e8fe8,26963808902597776881025.000000000


---


**Id:** G

**Description:** Calculates the average of the top 10 balances in quadrillions (1e15), rounding to two decimals—this is the final result of the main question.

**Results:**


,average_balance_trillion
0,26327002.23


---

In [10]:
generated_dag_queries_path = "/Users/stalaei/Desktop/Projects/Spider2/logs/generated_queries/logs/decomposed_sqls/test/gemini-1.5-pro-002-20250129-120811/sf_bq012.json"
generated_dag_queries = json.load(open(generated_dag_queries_path, 'r', encoding='utf-8'))
generated_dag_queries[0].keys()

dict_keys(['id', 'description', 'equivalent_natural_question', 'sql_query', 'dag_dependencies', 'generated_queries', 'sql_meta_data_info'])

In [20]:
our_query = """
SELECT
    "to_address" AS "address",
    "value" AS "value"
FROM "ETHEREUM_BLOCKCHAIN"."ETHEREUM_BLOCKCHAIN"."TRACES"
WHERE "to_address" IS NOT NULL
    AND "status" = 1
    AND ("call_type" NOT IN ('delegatecall', 'callcode', 'staticcall') OR "call_type" IS NULL)
"""

status, our_df = get_snowflake_sql_result(our_query, database_id, is_save=False)
print(f"The status of our query is {status}.")


generated_query = """
SELECT "to_address", SUM("value") AS total_ether_received
FROM "ETHEREUM_BLOCKCHAIN"."TRACES"
WHERE "status" = 1 AND ("call_type" IS NULL OR "call_type" = 'call') AND "to_address" IS NOT NULL
GROUP BY "to_address";
"""

status, generated_df = get_snowflake_sql_result(generated_query, database_id, is_save=False)
print(f"The status of generated query is {status}.")

generated_query_without_group_by = """
SELECT "to_address", "value"
FROM "ETHEREUM_BLOCKCHAIN"."TRACES"
WHERE "status" = 1 AND ("call_type" IS NULL OR "call_type" = 'call') AND "to_address" IS NOT NULL
"""

status, generated_df_without_group_by = get_snowflake_sql_result(generated_query_without_group_by, database_id, is_save=False)
print(f"The status of generated query without group by is {status}.")


In [21]:
compare_pandas_table(our_df, our_df)

1

In [22]:
compare_pandas_table(our_df, generated_df, ignore_order=True)

0

In [23]:
compare_pandas_table(our_df, generated_df_without_group_by, ignore_order=True)

1

In [9]:
print(f"The result of comparing our query and generated query is {compare_pandas_table(our_df, generated_df)}.")
print(f"The result of comparing our query and generated query without group by is {compare_pandas_table(our_df, generated_df_without_group_by)}.")

In [16]:
from IPython.display import display, Markdown

for node in generated_dag_queries:
    # Display node info
    display(Markdown(f"""
**Id:** {str(node.get('id', 'No ID found'))}

**Question:** {str(node.get('equivalent_natural_question', 'No question found'))}

**Description:** {str(node.get('description', 'No description found'))}

**Original Query Results:**
"""))
    
    # Execute and display original query results
    if "sql_query" not in node:
        display(Markdown("Error: No SQL query found in node"))
        continue

    display(Markdown(f"""
**Original Query:**
```sql
{node["sql_query"]}
```
"""))
        
    status, orig_df = get_snowflake_sql_result(node["sql_query"], database_id, is_save=False)
    if status:
        display(orig_df.head())
    else:
        display(Markdown(f"Error executing original query: {orig_df}"))
    
    # Loop through candidates
    display(Markdown("\n**Generated Query Results:**"))
    
    for i, candidate in enumerate(node['generated_queries']['candidates']):
        display(Markdown(f"\nCandidate {i+1}:"))
        generated_query = candidate.get('generated_query')
        
        if not generated_query:
            display(Markdown("No generated query found"))
            continue
            
        display(Markdown(f"""
**Generated Query:**
```sql
{generated_query}
```
"""))
        status, gen_df = get_snowflake_sql_result(generated_query, database_id, is_save=False)
        if status:
            display(gen_df.head())
            # Compare results
            if orig_df.equals(gen_df):
                display(Markdown("✅ Results match original query"))
            else:
                display(Markdown("❌ Results differ from original query"))
        else:
            display(Markdown(f"Error executing generated query: {gen_df}"))
    
    display(Markdown("---"))



**Id:** A

**Question:** Which addresses on Ethereum have received Ether in straightforward, successful transactions (not using special call types), and how much Ether did each receive in total?

**Description:** Filters the Ethereum TRACES table for successful transactions (status = 1) with to_address not null, excluding call types (delegatecall, callcode, staticcall). Each qualifying row represents Ether credited to the to_address.

**Original Query Results:**



**Original Query:**
```sql
SELECT
    "to_address" AS "address",
    "value" AS "value"
  FROM "ETHEREUM_BLOCKCHAIN"."ETHEREUM_BLOCKCHAIN"."TRACES"
  WHERE "to_address" IS NOT NULL
    AND "status" = 1
    AND ("call_type" NOT IN ('delegatecall', 'callcode', 'staticcall') OR "call_type" IS NULL)
```


,address,value
0,0xea674fdde714fd979de3edf0f56aa9716b898ec8,5000000000000000000.000000000
1,0xb2930b35844a230f00e51431acae96fe543a0347,5000000000000000000.000000000
2,0x2a65aca4d5fc5b5c859090a6c34d164135398226,3000000000000000000.000000000
3,0x52bc44d5378309ee2abf1539bf71de1b7d7be3b5,5000000000000000000.000000000
4,0xea674fdde714fd979de3edf0f56aa9716b898ec8,5000000000000000000.000000000



**Generated Query Results:**


Candidate 1:

...


Candidate 2:


**Generated Query:**
```sql

SELECT "from_address", "receipt_gas_used" * "gas_price" AS gas_fee_per_transaction, SUM("receipt_gas_used" * "gas_price") OVER (PARTITION BY "from_address") AS total_gas_fee_spending
FROM "ETHEREUM_BLOCKCHAIN"."TRANSACTIONS";

```


,from_address,GAS_FEE_PER_TRANSACTION,TOTAL_GAS_FEE_SPENDING
0,0x9ddc772bbfd702d2a1e246cd6dd26a312f15eef7,786030000000000,30380200000000000
1,0x9ddc772bbfd702d2a1e246cd6dd26a312f15eef7,786030000000000,30380200000000000
2,0x4fbda70fe3047a2764aeb621cb7857ca6cc385c5,471618000000000,471618000000000
3,0x2b5634c42055806a59e9107ed44d43c426e58258,789870000000000,376448475635698409
4,0x0681d8db095565fe8a346fa0277bffde9c0edbbf,2106320000000000,1858995040000000000


❌ Results differ from original query

---